# Lab Day 10 — Kaggle E0–E4
Read KAGGLE.md before Save Version. Enable Internet, GPU, attach the dataset, and grant GITHUB_TOKEN secret access. Push this implementation to main first. This notebook runs all required experiments; complete REPORT.md locally after collecting real results.

## 0. Environment, clone and preflight

In [ ]:
import os, sys, subprocess, json, urllib.request
from pathlib import Path
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/minhduong814/K4-Track-4-Day10-VisionLanguageModel.git"
SOURCE_REF = "main"  # Push local implementation here before Save Version.
RESULT_BRANCH = "kaggle-results-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
REPO = Path("/kaggle/working/lab-repo")
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
# Check permissions without printing the token or response body.
request = urllib.request.Request("https://api.github.com/repos/minhduong814/K4-Track-4-Day10-VisionLanguageModel",
    headers={"Authorization": "Bearer " + GITHUB_TOKEN, "Accept": "application/vnd.github+json"})
with urllib.request.urlopen(request, timeout=30) as response:
    permissions = json.load(response).get("permissions", {})
assert permissions.get("push"), "GitHub token must have write access to this repository"
DATASET_ROOTS = [p.parent for p in Path("/kaggle/input").rglob("labels-YOLO")
                 if p.is_dir() and (p.parent / "images").is_dir()]
assert len(DATASET_ROOTS) == 1, "Attach exactly one road-damage dataset with images/ and labels-YOLO/"
assert not REPO.exists(), "Use a fresh Save & Run All session; lab-repo already exists"
os.environ["HF_HOME"] = "/tmp/lab-hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run(["git", "clone", "--branch", SOURCE_REF, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
assert (REPO / "student/pipeline.py").exists(), "Push student implementation to GitHub before running"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt", "pytest"], check=True)
subprocess.run(["nvidia-smi", "-L"], check=True)
import torch
assert torch.cuda.is_available(), "Enable Kaggle GPU"
assert torch.cuda.get_device_properties(0).total_memory >= 15 * 1024**3, "GPU needs approximately 16 GB VRAM"
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
print("Preflight passed; result branch:", RESULT_BRANCH)

## 1. Prepare dataset and session splits

In [ ]:
subprocess.run([sys.executable, "scripts/prepare_data.py", "--source", str(DATASET_ROOTS[0])], check=True)
print((REPO / "data/splits/meta.json").read_text())

## 2. YOLO to JSON labels — student/pipeline.py: answer(), 0–1000 coordinates

## 3. E0 zero-shot — fresh Qwen3-VL-4B NF4, vision and lm_head unquantized

## 4. Training examples — chat-prefix assertion, answer-only loss, 220 image-token assertion

## 5. E1/E2/E3 — assert 252/348 LoRA modules; fp32 trainable projector for E2/E3

## 6. Train — 300 steps, accumulation 8, warmup/cosine, finite loss and gradient checks, checkpoints

## 7. Reload adapters/projector and score — validation selects E4 base; test evidence uses original scorer

## 8. Execute E0–E4 and push results — E4 rank 4 vs 64; E5 excluded

In [ ]:
# All experiments are executed in a subprocess to isolate CUDA memory from notebook state.
# 5 training runs: E1, E2, E3, E4 rank4, E4 rank64. E0 is zero-shot.
import traceback
from student.github_sync import push_results
pipeline_error = None
push_error = None
log_path = REPO / "report/runs/pipeline.log.txt"
log_path.parent.mkdir(parents=True, exist_ok=True)
try:
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen([sys.executable, "-u", "student/pipeline.py", "--batch-size", "1", "--grad-accum", "8"],
            cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        code = proc.wait()
    if code:
        raise RuntimeError(f"Pipeline exited {code}; see report/runs/pipeline.log.txt")
except Exception as exc:
    pipeline_error = exc
    (REPO / "report/runs/status.json").write_text(json.dumps({"pipeline": "failed", "error": str(exc)}))
finally:
    try:
        push_results(REPO, GITHUB_TOKEN, RESULT_BRANCH)
    except Exception as exc:
        push_error = exc
        print("GitHub push failed; download report/ from Kaggle Output.")
    GITHUB_TOKEN = None
if pipeline_error:
    raise pipeline_error
if push_error:
    raise push_error
print("Completed. Pull branch", RESULT_BRANCH, "at local to finish REPORT.md.")